# Crawl API

Bright Data Crawl API — extract structured page content from any URL. Each crawled record carries multiple output formats at once (markdown, html2text, page_html, ld_json, page_title, timestamp, url, input).

Two paths:
- **Sync** — `client.crawler.crawl(urls)` — one call, blocks until done.
- **Async** — `client.crawler.trigger(urls)` → `status(id)` → `download(id)` — snapshot lifecycle, good for big jobs.


## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

API_TOKEN = os.getenv("BRIGHTDATA_API_TOKEN")
if not API_TOKEN:
    raise ValueError("Set BRIGHTDATA_API_TOKEN in .env file")

print(f"API Token: {API_TOKEN[:10]}...{API_TOKEN[-4:]}")
print("Setup complete!")


## Initialize Client

In [ ]:
from brightdata import BrightDataClient

# timeout=120 so multi-URL crawls (~2s per URL via /scrape) have room to
# complete without hitting the 30s default.
client = BrightDataClient(token=API_TOKEN, timeout=120)

print("Client initialized")
print(f"Crawler dataset_id: {client.crawler.DATASET_ID}")


---
## Test 1: Sync crawl (single URL)

One round-trip — the API returns the parsed result inline.


In [ ]:
URL = "https://example.com"

async with client:
    result = await client.crawler.crawl(urls=URL)

print(f"Success: {result.success}")
print(f"Pages: {result.page_count}")
print(f"Error: {result.error}")

if result.success and result.data:
    rec = result.data[0]
    print(f"\n--- Record keys ---")
    print(sorted(rec.keys()))
    print(f"\nurl:        {rec.get('url')}")
    print(f"page_title: {rec.get('page_title')}")
    print(f"timestamp:  {rec.get('timestamp')}")
    md = rec.get('markdown') or ''
    print(f"\nmarkdown ({len(md)} chars):\n{md[:300]}..." if len(md) > 300 else f"\nmarkdown:\n{md}")


---
## Test 2: Sync crawl (multiple URLs)

Pass a list — the API crawls them all in one POST and returns one record per URL.


In [ ]:
URLS = [
    "https://example.com",
    "https://example.org",
]

async with client:
    result = await client.crawler.crawl(urls=URLS)

print(f"Success: {result.success}  Pages: {result.page_count}")

if result.success:
    for i, rec in enumerate(result.data, 1):
        title = rec.get('page_title') or '(no title)'
        print(f"\n{i}. {rec.get('url')}")
        print(f"   title:    {title}")
        print(f"   markdown: {len(rec.get('markdown') or '')} chars")
        print(f"   html:     {len(rec.get('page_html') or '')} chars")
else:
    print(f"Error: {result.error}")


---
## Test 3: Async path — trigger / status / download

Use this for big jobs where you don't want to keep one HTTP connection open. `trigger()` returns immediately with a `snapshot_id`; `status()` checks progress; `download()` polls until ready and fetches.


In [ ]:
URL = "https://example.com"

async with client:
    # 1. Trigger
    job = await client.crawler.trigger(urls=URL)
    print(f"Triggered: snapshot_id = {job.snapshot_id}")
    print(f"Sent at:   {job.trigger_sent_at}")

    # 2. Check status (one-shot — doesn't block)
    s = await client.crawler.status(job.snapshot_id)
    print(f"\nStatus right after trigger: {s}")

    # 3. Download — polls every 3s until ready, then fetches
    result = await client.crawler.download(
        job.snapshot_id,
        poll_interval=3,
        poll_timeout=180,
    )

print(f"\nDownloaded: success={result.success}  pages={result.page_count}")
if result.success and result.data:
    rec = result.data[0]
    print(f"snapshot_id: {result.snapshot_id}")
    print(f"page_title:  {rec.get('page_title')}")
    print(f"markdown:    {len(rec.get('markdown') or '')} chars")
elif result.error:
    print(f"Error: {result.error}")


---
## Summary

### Methods

| Method | Endpoint | Returns |
|---|---|---|
| `client.crawler.crawl(urls)` | `POST /datasets/v3/scrape` | `CrawlResult` (inline, blocks ~2s/URL) |
| `client.crawler.trigger(urls)` | `POST /datasets/v3/trigger` | `CrawlJob` with `snapshot_id` |
| `client.crawler.status(snapshot_id)` | `GET /datasets/v3/progress/<id>` | status string (`running` / `ready` / `failed` …) |
| `client.crawler.download(snapshot_id)` | polls then `GET /datasets/v3/snapshot/<id>` | `CrawlResult` |

### Record shape (each entry in `result.data`)

| Field | Description |
|---|---|
| `url` | The crawled URL (final, post-redirects) |
| `markdown` | Markdown rendering of the page content |
| `html2text` | Plain-text rendering |
| `page_html` | Full HTML of the page |
| `ld_json` | Extracted JSON-LD structured data, if any |
| `page_title` | `<title>` tag |
| `timestamp` | When the page was fetched |
| `input` | The original input dict passed to the API |

### Errors

- Invalid URLs / empty list → `ValidationError` raised immediately.
- HTTP / network errors → returned in `result.error` with `success=False`.
- `download()` snapshot timeout → returned in `result.error`, never raises `TimeoutError`.
